Dependencies

In [30]:
import pandas as pd
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

Load data

In [31]:
load_dotenv()

df = pd.read_csv("../data/transformed_news_data.csv")

df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_name
0,94624618,2026-04-19,Prediction: It's Not Too Late to Buy Broadcom ...,finance.yahoo.com,"['Broadcom', 'Broadcom Stock', 'Communication ...",2026-04-19,"['avgo', 'brcm', 'intc', 'iqnt', 'meta', 'nflx...",Sunday
1,94622595,2026-04-19,This Stock Will Be More Profitable Than Amazon...,finance.yahoo.com,"['Amazon', 'Communication Services', 'Compound...",2026-04-19,"['amd', 'amzn', 'intc', 'iqnt', 'meta', 'mu', ...",Sunday
2,94621400,2026-04-19,Option Traders Chasing Torrid Stock Rally Turn...,bloomberg.com,"['Financial Services', 'Stock', 'Technology', ...",2026-04-19,"['jpm', 'meta', 'msft']",Sunday
3,94612855,2026-04-18,Broadcom (AVGO) Ranks Among Unrivaled Stocks o...,finance.yahoo.com,"['Blue Chip Stocks', 'Broadcom Inc.', 'Inc.', ...",2026-04-18,"['avgo', 'brcm', 'meta']",Saturday
4,94612626,2026-04-18,Meta Platforms (META) Expands AI Chip Deal wit...,insidermonkey.com,"['Stock', 'Technology']",2026-04-18,['meta'],Saturday


Enrich with labels and score

In [32]:
client = InferenceClient(
    provider="hf-inference",
    api_key=os.getenv('hugging_face_token'),
)

results = client.text_classification(
    df['title'].to_list(),
    model="ProsusAI/finbert",
)

# Generated with AI (Gemini 3 fast)
df['sentiment_label'] = [res.label for res in results]
df['sentiment_score'] = [res.score for res in results]

df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_name,sentiment_label,sentiment_score
0,94624618,2026-04-19,Prediction: It's Not Too Late to Buy Broadcom ...,finance.yahoo.com,"['Broadcom', 'Broadcom Stock', 'Communication ...",2026-04-19,"['avgo', 'brcm', 'intc', 'iqnt', 'meta', 'nflx...",Sunday,neutral,0.810057
1,94622595,2026-04-19,This Stock Will Be More Profitable Than Amazon...,finance.yahoo.com,"['Amazon', 'Communication Services', 'Compound...",2026-04-19,"['amd', 'amzn', 'intc', 'iqnt', 'meta', 'mu', ...",Sunday,positive,0.585464
2,94621400,2026-04-19,Option Traders Chasing Torrid Stock Rally Turn...,bloomberg.com,"['Financial Services', 'Stock', 'Technology', ...",2026-04-19,"['jpm', 'meta', 'msft']",Sunday,negative,0.485695
3,94612855,2026-04-18,Broadcom (AVGO) Ranks Among Unrivaled Stocks o...,finance.yahoo.com,"['Blue Chip Stocks', 'Broadcom Inc.', 'Inc.', ...",2026-04-18,"['avgo', 'brcm', 'meta']",Saturday,positive,0.770807
4,94612626,2026-04-18,Meta Platforms (META) Expands AI Chip Deal wit...,insidermonkey.com,"['Stock', 'Technology']",2026-04-18,['meta'],Saturday,positive,0.745573


In [33]:
df.to_csv('../data/sentiment_model_output.csv', index=False)